# VWAP Events 02 — touches, crosses and time to return (descriptive)

BTC and ETH, Binance USD-M perpetual 1m. Session VWAP anchored 08:00 UTC; 4H VWAP anchored 00/04/08/12/16/20 UTC (definitions as VWAP Events 01).

- **Touch:** bar range contains the VWAP known at the bar's open. Consecutive touch bars = one *episode*.
- **Cross:** close switches side of VWAP (within the same anchor window).
- **After a cross:** minutes until the next **touch** (price back at VWAP from the new side), until a **cross back**, and until a touch **after price first detached** (removes the trivial "still sitting on it" case). Crosses whose window resets first are censored; Kaplan-Meier medians account for that.
- **Placebo:** the same statistics for VWAPs anchored at unused times (session at 02/14/20 UTC; 4H offset by 1/2/3 h).

Descriptive only: no gates, no trading claim.

In [ ]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
pd.set_option('display.width',250);pd.set_option('display.max_columns',40)
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import vwap_events as ve, vwap_touch_stats as vt
importlib.reload(vt)
SYNTHETIC=os.getenv('VE02_SYNTHETIC','0')=='1'
ASSETS=['BTC','ETH']
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='ve02-');OUT=Path(tmp.name)/'02'
else:
    OUT=ROOT/'notebooks'/'VWAP_Events'/'02'
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
OUT.mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_vwap_touch_stats.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-400:]);assert chk.returncode==0,'tests failed'
print(vt.VERSION)

In [ ]:
CNT,SUM,EV=[],[],[]
for i,a in enumerate(ASSETS):
    t0=time.time()
    df=ve.synthetic_klines(n_days=120,seed=5+i) if SYNTHETIC else pd.read_parquet(BIN/f'{a}USDT_um_1m.parquet',columns=['open_time','high','low','close','volume','quote_volume'])
    c,s,e=vt.run_asset(df,a);CNT.append(c);SUM.append(s);EV.append(e)
    print(a,f'{df.open_time.min()} -> {df.open_time.max()}',f'{time.time()-t0:.0f}s')
CNT=pd.concat(CNT,ignore_index=True);SUM=pd.concat(SUM,ignore_index=True);EV=pd.concat(EV,ignore_index=True)
CNT.to_csv(OUT/'02_counts.csv',index=False);SUM.to_csv(OUT/'02_return_times.csv',index=False)

## 1. How often: touches and crosses

In [ ]:
show=CNT[CNT.kind=='real'][['asset','anchor','days','windows','touch_bars','touch_episodes','touch_episodes_per_window','crosses','crosses_up','crosses_down','crosses_per_window','crosses_per_day']]
display(show.round(2))
print('Real vs placebo anchors (touch episodes and crosses per window):')
display(CNT.pivot_table(index=['asset','anchor'],columns='kind',values=['touch_episodes_per_window','crosses_per_window']).round(2))

## 2. After a cross: time until price comes back

In [ ]:
s=SUM[(SUM.kind=='real')]
cols=['asset','anchor','measure','direction','n','share_returned_before_reset','median_min_if_returned','mean_min_if_returned','km_median_min','share_within_1m','share_within_5m','share_within_15m','share_within_60m']
for m in ('min_to_touch','min_to_return_after_detach' if False else 'min_return_after_detach','min_to_cross_back'):
    print(m);display(s[s.measure==m][cols].round(3))

## 3. Real vs placebo anchors

In [ ]:
rvp=vt.real_vs_placebo(SUM);rvp.to_csv(OUT/'02_real_vs_placebo.csv',index=False);display(rvp.round(3))

## 4. Distribution of return times after detaching (real anchors, minutes)

In [ ]:
q=EV[EV.kind=='real'].groupby(['asset','anchor'])['min_return_after_detach'].describe(percentiles=[.1,.25,.5,.75,.9]).round(1)
display(q);q.to_csv(OUT/'02_return_after_detach_quantiles.csv')
print('time of day of crosses (UTC hour, share):')
h=EV[EV.kind=='real'].assign(hour=lambda d:pd.to_datetime(d.t).dt.hour).groupby(['asset','anchor']).hour.value_counts(normalize=True).unstack().round(3)
display(h);h.to_csv(OUT/'02_cross_hour_share.csv')
(OUT/'02_manifest.json').write_text(json.dumps({'version':vt.VERSION,'assets':ASSETS,'synthetic':SYNTHETIC},indent=1))